# Marqo-FashionSigLIP × MAEVEN — gắn nhãn tự động và "phối với đồ đã có"

Bước 3 trong lộ trình học (`research/README.md`). Model biến **ảnh và chữ về thời trang thành vector cùng một không gian**. Hai thứ giống nhau thì vector gần nhau. Notebook thử ba việc:

1. **Gắn nhãn tự động** loại món, màu, chất liệu, phom, dịp mặc. Kết quả là bản nháp cho trường `style` ở Phase 0 của plan.
2. **"Phối với đồ mình đã có"**: đưa ảnh một món trong tủ của khách, tìm món MAEVEN đi cùng.
3. **Tìm bằng chữ**: gõ mô tả, tìm sản phẩm.

> **License:** Apache 2.0, được dùng thương mại. Chạy được cả trên CPU, nhưng GPU nhanh hơn.

**Kaggle:** bật *Internet*. Dùng chung dataset `maeven-tryon` của notebook CatVTON, thêm thư mục `wardrobe/` chứa ảnh vài món đồ bất kỳ (không bắt buộc).


In [ ]:
!pip install -q open_clip_torch

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
import open_clip
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "hf-hub:Marqo/marqo-fashionSigLIP"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL)
tokenizer = open_clip.get_tokenizer(MODEL)
model = model.to(DEVICE).eval()
print(DEVICE)

In [ ]:
@torch.no_grad()
def embed_images(paths):
    batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths]).to(DEVICE)
    return model.encode_image(batch, normalize=True).float().cpu()

@torch.no_grad()
def embed_texts(texts):
    tokens = tokenizer(texts).to(DEVICE)
    return model.encode_text(tokens, normalize=True).float().cpu()

def rank(image_vecs, labels, template="a photo of {}"):
    """Với mỗi ảnh, trả về các nhãn xếp theo điểm giống nhau (cosine)."""
    text_vecs = embed_texts([template.format(l) for l in labels])
    sims = image_vecs @ text_vecs.T
    probs = (100.0 * sims).softmax(dim=-1)  # như README của Marqo; chỉ dùng để xếp hạng giữa các nhãn
    return [
        sorted(zip(labels, p.tolist()), key=lambda x: -x[1])
        for p in probs
    ]

## 1. Nạp catalog
Ảnh lấy từ `garments/` của dataset. Nếu chưa có dataset thì upload ảnh trong `public/img/product/` của repo. Lưu ý: đó là ảnh stock, nên nhãn sẽ mô tả ảnh stock chứ không phải áo thật.

In [ ]:
import zipfile

# Tên và nhóm thật của sản phẩm: nhóm đã biết chắc, dùng để chấm model chứ không cần model đoán
CATALOG = {
    "mv-01": ("Áo sơ mi lanh mộc", "top"),
    "mv-02": ("Quần ống suông", "bottom"),
    "mv-03": ("Áo khoác không cổ", "outer"),
    "mv-04": ("Áo thun cổ tròn", "top"),
}
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def find_dataset(root=Path("/kaggle/input")):
    # Tìm thư mục chứa garments/ ở mọi độ sâu. Kaggle có thể gắn dataset sâu hơn một tầng,
    # hoặc giữ nguyên file zip; khi đó tự giải nén vào /kaggle/working/data.
    hits = sorted(p.parent for p in root.rglob("garments") if p.is_dir())
    if hits:
        return hits[0]
    for z in root.rglob("*.zip"):
        out = Path("/kaggle/working/data") / z.stem
        with zipfile.ZipFile(z) as f:
            f.extractall(out)
        hits = sorted(p.parent for p in out.rglob("garments") if p.is_dir())
        if hits:
            return hits[0]
    return None

def show_input_tree(root=Path("/kaggle/input"), limit=40):
    print("Không thấy thư mục garments/. Những gì đang có trong /kaggle/input:")
    for p in sorted(root.rglob("*"))[:limit]:
        print("  ", p)

DATA = find_dataset()
if DATA is None:
    show_input_tree()
    raise FileNotFoundError("Kiểm tra dataset đã được thêm qua 'Add Input' chưa")

items = {p.stem: p for p in sorted((DATA / "garments").iterdir())
         if p.suffix.lower() in IMG_EXT and p.stem in CATALOG}
assert items, f"{DATA / 'garments'} không có file nào tên mv-01 … mv-04"
skus = list(items)
print(f"Dataset: {DATA}")
catalog_vecs = embed_images([items[s] for s in skus])
print(f"{len(skus)} sản phẩm · vector {catalog_vecs.shape[1]} chiều")

## 2. Gắn nhãn tự động → bản nháp cho Phase 0

Model trả lời dạng *"ảnh này giống nhãn nào nhất trong danh sách"*, gọi là **zero-shot**: không cần train thêm. Nhãn viết bằng tiếng Anh vì model được train bằng tiếng Anh.

**Đây chỉ là bản nháp.** Người phải duyệt lại trước khi đưa vào `src/lib/data.ts`. Trong `scores`, điểm cao không có nghĩa là đúng, chỉ có nghĩa là model tự tin.

In [ ]:
ATTRS = {
    # nhãn → giá trị trong schema Phase 0
    "category": {"a button-up shirt": "top", "a crew neck t-shirt": "top", "a pair of trousers": "bottom",
                 "a jacket": "outer", "a pair of shorts": "bottom", "a sweater": "top"},
    "color": {"white clothing": "trắng", "off-white ecru clothing": "trắng ngà", "beige clothing": "be",
              "grey clothing": "xám", "black clothing": "đen", "navy clothing": "xanh navy",
              "olive green clothing": "xanh rêu", "brown clothing": "nâu", "light blue clothing": "xanh nhạt"},
    "material": {"linen fabric": "lanh", "cotton twill fabric": "bông dệt chéo", "cotton jersey knit": "bông dệt kim",
                 "hemp fabric": "gai dầu", "wool fabric": "len", "denim fabric": "denim"},
    "fit": {"slim fit clothing": "slim", "regular fit clothing": "regular", "relaxed loose fit clothing": "relaxed"},
    "occasion": {"office workwear": "đi làm", "casual weekend outfit": "cuối tuần",
                 "smart wedding guest outfit": "đám cưới", "travel outfit": "du lịch"},
}

draft = {}
for attr, mapping in ATTRS.items():
    for sku, ranked in zip(skus, rank(catalog_vecs, list(mapping))):
        d = draft.setdefault(sku, {"name": CATALOG[sku][0], "scores": {}})
        d["scores"][attr] = [(mapping[l], round(p, 3)) for l, p in ranked[:3]]
        top = [mapping[l] for l, _ in ranked[:2]] if attr in ("color", "occasion") else mapping[ranked[0][0]]
        d[attr] = top

for sku, d in draft.items():
    ok = "✓" if d["category"] == CATALOG[sku][1] else "✗"
    print(f"{sku} {d['name']:<20} {d['category']:<7}{ok} {', '.join(d['color']):<22} {d['material']:<14} {d['fit']:<8} {', '.join(d['occasion'])}")

Path("/kaggle/working/style_draft.json").write_text(json.dumps(draft, ensure_ascii=False, indent=2))

Cột ✓/✗ so loại món model đoán với loại thật. Tự kiểm tra thêm: mv-01 phải ra *lanh*, mv-02 *bông dệt chéo*, mv-04 *bông dệt kim*. Nếu sai thì vì sao? Có thể do ảnh stock, do nhãn viết chưa rõ, hoặc do model chưa phân biệt được hai chất liệu. Ghi lại vào nhật ký.

## 3. "Phối với đồ mình đã có"

Cách làm đơn giản nhất:
1. Đoán món của khách thuộc nhóm nào (`top` / `bottom` / `outer`).
2. Chỉ xét các món MAEVEN thuộc **nhóm khác**. Áo thì gợi ý quần, quần thì gợi ý áo.
3. Xếp hạng theo độ giống của vector ảnh, coi như thước đo gần đúng cho "cùng tông, cùng chất".

**Giới hạn cần hiểu:** *giống nhau* không có nghĩa là *hợp nhau*. Một chiếc quần be giống áo be nhất, nhưng chưa chắc là món đẹp nhất để phối. Muốn học "hợp nhau" thật sự thì cần dữ liệu outfit như **Polyvore**. Đó là bước tiếp theo nếu bạn muốn đi sâu.

In [ ]:
SLOT_LABELS = {"a shirt or t-shirt or sweater": "top", "a pair of trousers or shorts or jeans": "bottom",
               "a jacket or coat": "outer"}

def pair_with(path, k=2):
    vec = embed_images([path])
    slot = SLOT_LABELS[rank(vec, list(SLOT_LABELS))[0][0][0]]
    sims = (vec @ catalog_vecs.T)[0]
    candidates = [(s, float(sims[i])) for i, s in enumerate(skus) if CATALOG[s][1] != slot]
    return slot, sorted(candidates, key=lambda x: -x[1])[:k]

wardrobe = DATA / "wardrobe"
if wardrobe.is_dir():
    for p in sorted(wardrobe.iterdir()):
        if p.suffix.lower() not in IMG_EXT:
            continue
        slot, picks = pair_with(p)
        print(f"{p.name:<24} → nhóm {slot:<6} → " + ", ".join(f"{s} {CATALOG[s][0]} ({v:.2f})" for s, v in picks))
        display(Image.open(p).resize((160, int(160 * Image.open(p).height / Image.open(p).width))))
else:
    print("Chưa có thư mục wardrobe/: thêm vài ảnh đồ của bạn để thử.")

## 4. Tìm bằng chữ
Model chỉ hiểu tiếng Anh. Trong AI Stylist, Claude sẽ dịch câu hỏi tiếng Việt của khách sang mô tả tiếng Anh trước khi tìm.

In [ ]:
QUERIES = [
    "a breathable shirt for hot summer days",
    "smart trousers for the office",
    "a light layer to wear over a shirt",
    "a basic everyday t-shirt",
]
sims = embed_texts(QUERIES) @ catalog_vecs.T
for q, row in zip(QUERIES, sims):
    best = row.argsort(descending=True)[:2]
    print(f"{q:<44} → " + ", ".join(f"{skus[i]} ({row[i]:.2f})" for i in best))

## 5. Lưu vector
Để dùng lại về sau mà không phải chạy model lại, ví dụ để thử ở Phase 3 của plan.

In [ ]:
Path("/kaggle/working/catalog_embeddings.json").write_text(json.dumps(
    {"model": MODEL, "skus": skus, "vectors": np.round(catalog_vecs.numpy(), 5).tolist()}
))
print("→ /kaggle/working/style_draft.json, catalog_embeddings.json")